In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
MODEL_NAME = "sdadas/polish-gpt2-medium"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

if model.config.pad_token_id is None and model.config.eos_token_id is not None:
    model.config.pad_token_id = model.config.eos_token_id


Urządzenie: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

In [ ]:
def score_text(text: str, model, tokenizer, device: str) -> float:
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    # przesuwamy: logits[:, :-1, :] przewidują input_ids[:, 1:]
    logits = logits[:, :-1, :]
    target_ids = input_ids[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)

    # wybieramy log-p dla właściwych tokenów
    token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)

    total_log_prob = token_log_probs.sum().item()
    return total_log_prob


In [ ]:
def disambiguate_line(line: str, model, tokenizer, device: str, beam_size: int = 5) -> str:
    positions = line.strip().split()
    variant_lists = [pos.split("|") for pos in positions]

    # Każdy beam to (lista_słów, log_prob)
    beams = [([], 0.0)]

    for pos_idx, variants in enumerate(variant_lists):
        new_beams = []

        for words_so_far, score_so_far in beams:
            for v in variants:
                candidate_words = words_so_far + [v]
                candidate_text = " ".join(candidate_words)

                lp = score_text(candidate_text, model, tokenizer, device)

                new_beams.append((candidate_words, lp))

        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_size]

    best_words, best_score = beams[0]
    return " ".join(best_words)


In [ ]:
example = """wprost|wyprosty|wyprostu|wyprost uwielbiała|wielbił|wielbiła|uwielbił|wielbiło|uwielbiał|uwielbiało|uwielbiały
słuchać|osłuchać|słychać|usłuchać o|i|e|a|ó|ę|y|ą|u
wartościach własnych|owłosionych macierzy|mocarz|macierzą|macierze|mocarza|mocarze|mocarzy|macierz"""

for line in example.splitlines():
    print("WEJŚCIE: ", line)
    out = disambiguate_line(line, model, tokenizer, device, beam_size=5)
    print("WYJŚCIE:", out)
    print()


WEJŚCIE:  wprost|wyprosty|wyprostu|wyprost uwielbiała|wielbił|wielbiła|uwielbił|wielbiło|uwielbiał|uwielbiało|uwielbiały
WYJŚCIE: wprost uwielbiał

WEJŚCIE:  słuchać|osłuchać|słychać|usłuchać o|i|e|a|ó|ę|y|ą|u
WYJŚCIE: słuchać o

WEJŚCIE:  wartościach własnych|owłosionych macierzy|mocarz|macierzą|macierze|mocarza|mocarze|mocarzy|macierz
WYJŚCIE: wartościach własnych macierzy



In [ ]:
def disambiguate_file(path, model, tokenizer, device, beam_size=5):
    results = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sent = disambiguate_line(line, model, tokenizer, device, beam_size)
            results.append(sent)
    return results

wyniki = disambiguate_file("zdania_z_wariantami.txt", model, tokenizer, device)
for s in wyniki: print(s)


parlament zdecydował jednak inaczej i przyjął w ustawie z dnia 28.06.1996 r. jednoinstancyjne postępowanie orzeczniczo-lekarskie .
po kampanii wrześniowej 1.dlek raportował 77 czołgów l.t.m.35 utraconych ( wraz z wozem dowodzenia ) i 52 uszkodzone lub zepsute , lecz ostatecznie po naprawach straty bezpowrotne ograniczyły się do 7 czołgów i w lutym 1940 roku posiadał 195 czołgów na stanie .
może ten przypadek odstraszy innych kłusowników od tego procederu .
ulgi i przywileje są w stosunkach między państwami zwykle wzajemne , ambasador ma tu niewielkie pole działania .
pośród wzgórza do strony południowej do państwa wpływał wierzbiak , który znany był już ze złotonośnych piasków .
tutaj możecie zobaczyć , jak wygląda to w praktyce , na mapie są wyszczególnione miejsca objęte programem .
przykładowo : w sferze oświaty działania placówek dyplomatyczno-konsularnych obejmują m.in . pomoc w zakładaniu punktów nauczania języka polskiego , fundowanie nagród dla najlepszych uczniów i nauczycieli

In [ ]:
PATH_WITH_VARIANTS = "zdania_z_wariantami.txt"

def evaluate_from_predictions(path, predictions):
    total_positions = 0
    correct_positions = 0

    total_sentences = 0
    correct_sentences = 0

    examples = []

    pred_idx = 0

    with open(path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            if pred_idx >= len(predictions):
                print("Uwaga: skończyły się predykcje wcześniej niż linie w pliku!")
                break

            pred_sentence = predictions[pred_idx]
            pred_idx += 1

            pos_strings = line.split()
            variant_lists = [pos.split("|") for pos in pos_strings]

            gold_sequence = [variants[0] for variants in variant_lists]

            pred_tokens = pred_sentence.split()

            L = min(len(pred_tokens), len(gold_sequence))
            pred_tokens = pred_tokens[:L]
            gold_sequence = gold_sequence[:L]

            sentence_correct = True
            for g, p in zip(gold_sequence, pred_tokens):
                total_positions += 1
                if g == p:
                    correct_positions += 1
                else:
                    sentence_correct = False

            total_sentences += 1
            if sentence_correct and len(pred_tokens) == len(variant_lists):
                correct_sentences += 1

            if len(examples) < 5:
                examples.append({
                    "line": line,
                    "gold": " ".join(gold_sequence),
                    "pred": " ".join(pred_tokens),
                })

    if pred_idx < len(predictions):
        print(f"Uwaga: w 'predictions' jest {len(predictions)} zdań, "
              f"a wykorzystaliśmy tylko {pred_idx} (w pliku mniej niepustych linii).")

    token_acc = correct_positions / total_positions if total_positions > 0 else 0.0
    sent_acc = correct_sentences / total_sentences if total_sentences > 0 else 0.0

    print("\n=== WYNIKI (z gotowych predykcji) ===")
    print("Liczba zdań:", total_sentences)
    print("Liczba pozycji (słów):", total_positions)
    print("Accuracy per słowo:", token_acc)
    print("Accuracy per zdanie:", sent_acc)

    print("\nPrzykładowe (gold vs pred):")
    for ex in examples:
        print("-" * 40)
        print("ORIG:", ex["line"])
        print("GOLD:", ex["gold"])
        print("PRED:", ex["pred"])

    return token_acc, sent_acc


In [ ]:
token_acc, sent_acc = evaluate_from_predictions(PATH_WITH_VARIANTS, wyniki)
print("\nAccuracy per słowo:", token_acc)
print("Accuracy per zdanie:", sent_acc)



=== WYNIKI (z gotowych predykcji) ===
Liczba zdań: 300
Liczba pozycji (słów): 6548
Accuracy per słowo: 0.9589187538179597
Accuracy per zdanie: 0.5333333333333333

Przykładowe (gold vs pred):
----------------------------------------
ORIG: parlament|parlamentem|parlamentów|parlamenty|parlamencie zdecydował|zdecyduje|zdecydowałaby|zdecydujecie|zdecydowali|zdecydowała jednak inaczej i przyjął|przyjąć|przyjąłby|przyjmiecie|przyjęła|przyjęli w ustawie|ustawom|ustawami|ustawa|ustawą|ustawach z dnia|dzień|dniom|dniami|dni|dniach 28.06.1996 r. jednoinstancyjne|jednoinstancyjnym|jednoinstancyjny|jednoinstancyjnego postępowanie|postępowaniom|postępowania|postępowaniu|postępowań orzeczniczo-lekarskie .
GOLD: parlament zdecydował jednak inaczej i przyjął w ustawie z dnia 28.06.1996 r. jednoinstancyjne postępowanie orzeczniczo-lekarskie .
PRED: parlament zdecydował jednak inaczej i przyjął w ustawie z dnia 28.06.1996 r. jednoinstancyjne postępowanie orzeczniczo-lekarskie .
-------------------------